# Nike Global — Exploratory Data Analysis
Quick look at the data and data quality checks.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Load Data

In [ ]:
FILE = 'datasets/Global_Nike.csv'

df = pd.read_csv(FILE)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/Nike_Global.csv'

## 2. Schema & Data Types

In [ ]:
schema = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notna().sum(),
    'null': df.isna().sum(),
    'null_%': (df.isna().mean() * 100).round(2),
    'unique': df.nunique(),
    'sample': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
})
schema

## 3. Data Quality Checks

### 3.1 Duplicates

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Fully duplicate rows: {n_dupes} ({n_dupes/len(df)*100:.2f}%)')
if n_dupes > 0:
    print('\nSample duplicate rows:')
    display(df[df.duplicated(keep=False)].head(6))

### 3.2 Missing Values

In [ ]:
missing = schema[schema['null'] > 0][['null', 'null_%']].sort_values('null_%', ascending=False)

if missing.empty:
    print('No missing values found.')
else:
    print(f'{len(missing)} columns have missing values:\n')
    display(missing)

    fig, ax = plt.subplots(figsize=(8, max(3, len(missing) * 0.4)))
    ax.barh(missing.index, missing['null_%'], color='#e87722')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Column')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

### 3.3 Numeric Columns — Descriptive Statistics & Outliers

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
print(f'Numeric columns ({len(num_cols)}): {num_cols}\n')

if num_cols:
    desc = df[num_cols].describe().T
    desc['IQR'] = desc['75%'] - desc['25%']
    desc['outlier_low']  = desc['25%'] - 1.5 * desc['IQR']
    desc['outlier_high'] = desc['75%'] + 1.5 * desc['IQR']
    display(desc)

In [ ]:
# Count IQR outliers per numeric column
if num_cols:
    outlier_counts = {}
    for col in num_cols:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        mask = (df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)
        outlier_counts[col] = mask.sum()

    outlier_df = pd.Series(outlier_counts, name='outlier_count').to_frame()
    outlier_df['outlier_%'] = (outlier_df['outlier_count'] / len(df) * 100).round(2)
    outlier_df = outlier_df[outlier_df['outlier_count'] > 0].sort_values('outlier_%', ascending=False)

    if outlier_df.empty:
        print('No IQR outliers detected.')
    else:
        print('Columns with IQR outliers:')
        display(outlier_df)

### 3.4 Categorical Columns — Cardinality & Top Values

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}\n')

for col in cat_cols:
    n_unique = df[col].nunique()
    top = df[col].value_counts(dropna=False).head(10)
    print(f'--- {col} ({n_unique} unique values) ---')
    display(top.to_frame('count').assign(pct=(top / len(df) * 100).round(2)))
    print()

### 3.5 Date/Time Columns

In [ ]:
# Auto-detect likely date columns
date_candidates = [c for c in df.columns if any(kw in c.lower() for kw in ['date', 'time', 'year', 'month', 'week'])]

if not date_candidates:
    print('No obvious date columns detected by name. Check manually if needed.')
else:
    for col in date_candidates:
        try:
            parsed = pd.to_datetime(df[col], errors='coerce')
            n_parsed = parsed.notna().sum()
            print(f'{col}: {n_parsed:,}/{len(df):,} parsed as dates | range: {parsed.min()} → {parsed.max()}')
        except Exception as e:
            print(f'{col}: could not parse — {e}')

## 4. Distributions of Numeric Columns

In [ ]:
if num_cols:
    n = len(num_cols)
    ncols = 2
    nrows = (n + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5))
    axes = axes.flatten()

    for i, col in enumerate(num_cols):
        data = df[col].dropna()
        axes[i].hist(data, bins=40, color='#111111', edgecolor='white', linewidth=0.3)
        axes[i].set_title(col, fontsize=10)
        axes[i].set_ylabel('Count')
        axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Numeric Column Distributions', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 5. Correlation Matrix (Numeric Columns)

In [ ]:
if len(num_cols) > 1:
    corr = df[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))

    fig, ax = plt.subplots(figsize=(max(6, len(num_cols)), max(5, len(num_cols) - 1)))
    im = ax.imshow(corr.where(~mask, other=np.nan), cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(len(num_cols)))
    ax.set_yticks(range(len(num_cols)))
    ax.set_xticklabels(num_cols, rotation=45, ha='right')
    ax.set_yticklabels(num_cols)

    for i in range(len(num_cols)):
        for j in range(len(num_cols)):
            if not mask[i, j]:
                ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)

    ax.set_title('Pearson Correlation (lower triangle)')
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 numeric columns for correlation.')

## 6. Quality Summary

In [ ]:
print('=' * 50)
print('DATA QUALITY SUMMARY')
print('=' * 50)
print(f'Rows:              {len(df):,}')
print(f'Columns:           {len(df.columns)}')
print(f'Duplicate rows:    {df.duplicated().sum():,}')
print(f'Columns w/ nulls:  {(df.isna().any()).sum()}')
print(f'Total null cells:  {df.isna().sum().sum():,} ({df.isna().mean().mean()*100:.2f}% of all cells)')
print(f'Numeric columns:   {len(num_cols)}')
print(f'Categorical cols:  {len(cat_cols)}')
print(f'Date candidates:   {len(date_candidates)}')